In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

np.random.seed(123)


# 1. Parameters

beta = 0.002
alpha = 0
income_mean = 50000
income_sd = 10000
eps_sd = 100

sigma_u_values = [0, 5000, 15000]
n_values = [250, 1000, 5000]

num_sims = 1000

# 2. Simulation

results = []

for n in n_values:

    for sigma_u in sigma_u_values:

        betas = []
        ses = []

        for _ in range(num_sims):

            # 3. Generate true income
            income = np.random.normal(
                income_mean,
                income_sd,
                n
            )

            # 4. Generate measurement error
            u = np.random.normal(
                0,
                sigma_u,
                n
            )

            # 5. Observed income with measurement error
            income_star = income + u

            # 6. Generate error term
            eps = np.random.normal(
                0,
                eps_sd,
                n
            )

            # 7. Generate test scores using true income
            test_score = (
                alpha
                + beta * income
                + eps
            )

            # 8. Run OLS using income measured with error
            X = sm.add_constant(income_star)

            model = sm.OLS(
                test_score,
                X
            ).fit()

            # 9. Store estimated coefficient and standard error
            betas.append(model.params[1])
            ses.append(model.bse[1])

        # 10. Theoretical expected value
        # Measurement error causes attenuation bias:
        # E[beta*] = beta * Var(income) /
        #           [Var(income) + Var(measurement error)]

        expected_beta = (
            beta
            * (income_sd**2)
            / (income_sd**2 + sigma_u**2)
        )

        # 11. Store results
        results.append({
            "n": n,
            "sigma_u": sigma_u,
            "Expected E[beta*]": expected_beta,
            "Avg Estimated beta*": np.mean(betas),
            "Avg OLS SE": np.mean(ses)
        })


# 12. Convert results to DataFrame

df_results = pd.DataFrame(results)

# 13. Save results to output folder
df_results.to_csv("../output/q2_results.csv", index=False)

# 14. Display results
df_results

,n,sigma_u,Expected E[beta*],Avg Estimated beta*,Avg OLS SE
0,250,0,0.002000,0.001998,0.000635
1,250,5000,0.001600,0.001575,0.000569
2,250,15000,0.000615,0.000611,0.000357
3,1000,0,0.002000,0.002003,0.000317
4,1000,5000,0.001600,0.001586,0.000284
5,1000,15000,0.000615,0.000610,0.000178
6,5000,0,0.002000,0.001997,0.000141
7,5000,5000,0.001600,0.001601,0.000127
8,5000,15000,0.000615,0.000613,0.000080
